In [1]:
import gzip
import json
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [ ]:
def load_json_gzp(path: str) -> pd.DataFrame:
    records = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)

meta_path = "data/meta-Virginia.json.gz"
review_path = "data/review-Virginia.json.gz"

df_meta = load_json_gzp(meta_path)
df_review = load_json_gzp(review_path)

Keep only business id, avg rating, and total # of reviews. Also drops rows with missing values

In [ ]:
biz = df_meta[["gmap_id", "avg_rating", "num_of_reviews"]].dropna()

biz["num_of_reviews"] = biz["num_of_reviews"].astype(int)
biz["avg_rating"] = biz["avg_rating"].astype(float)


In [ ]:
#X-axis labels and bins
bins = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, biz["num_of_reviews"].max() + 1]
labels = [f"{bins[i]}–{bins[i+1]-1}" for i in range(len(bins)-1)]

biz["review_bin"] = pd.cut(biz["num_of_reviews"], bins=bins, labels=labels, right=False)

#Calculate statistics for each bin
stability_table = (biz.groupby("review_bin").agg(
        n_businesses=("gmap_id", "count"),
        rating_std=("avg_rating", "std"),
        rating_var=("avg_rating", "var"),
        mean_rating=("avg_rating", "mean"),
    ).reset_index())

#Is this alright?
print(stability_table)

  review_bin  n_businesses  rating_std  rating_var  mean_rating
0        1–1          4228    1.223072    1.495906     4.271949
1        2–4         10213    0.875691    0.766835     4.282415
2        5–9         19405    0.722017    0.521309     4.263679
3      10–19         15062    0.661144    0.437111     4.256619
4      20–49         21531    0.620459    0.384970     4.268069
5      50–99         16440    0.567684    0.322265     4.286089
6    100–199         12988    0.492077    0.242140     4.310371
7    200–499         12316    0.451288    0.203661     4.282941
8    500–999          5013    0.410508    0.168517     4.214981
9  1000–9998          2477    0.381416    0.145479     4.251756


C:\Users\aiden\AppData\Local\Temp\ipykernel_27820\3415428455.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  biz.groupby("review_bin")


In [20]:
#Change in std between bins
stability_table["delta_std"] = stability_table["rating_std"].diff().abs()
stability_table

epsilon = 0.02  # ratings barely change beyond this
stable_bins = stability_table[stability_table["delta_std"] < epsilon]
#Weird output
print(stable_bins)


Empty DataFrame
Columns: [review_bin, n_businesses, rating_std, rating_var, mean_rating, delta_std]
Index: []


In [ ]:
#40+ reviews is stable (good marker?)
biz_stable = biz[biz["num_of_reviews"] >= 40]

In [ ]:
#Scale big numbers down because comparing was hard
biz["weight"] = np.log1p(biz["num_of_reviews"])

In [ ]:
#Makes grid over VA. Gemini helped heavily with this
def add_geo_cells(df, lat_col="latitude", lon_col="longitude", cell_size=0.1):
    #get used to using copy more and understanding it
    d = df.copy()
    d["lat_cell"] = (d[lat_col] // cell_size) * cell_size
    d["lon_cell"] = (d[lon_col] // cell_size) * cell_size
    d["geo_cell"] = d["lat_cell"].astype(str) + "_" + d["lon_cell"].astype(str)
    return d

df_meta_cells = add_geo_cells(df_meta)


In [ ]:
#Calculate statistics for each "grid"
cell_features = (
    df_meta_cells.groupby("geo_cell").agg(
        n_businesses=("gmap_id", "nunique"),
        avg_reviews_per_biz=("num_of_reviews", "mean"),
        median_reviews_per_biz=("num_of_reviews", "median"),
        pct_high_review_biz=("num_of_reviews", lambda x: (x >= 100).mean()),
        avg_rating=("avg_rating", "mean"),
    ).reset_index())


In [ ]:
if "categories" in df_meta_cells.columns:
    cat_div = (
        df_meta_cells.explode("categories")
        .groupby("geo_cell")["categories"]
        .nunique()
        .rename("category_diversity")
    )
    #Come back to this
    cell_features = cell_features.merge(cat_div, on="geo_cell", how="left")

In [ ]:
#density metrics
features = ["n_businesses", "avg_reviews_per_biz", "pct_high_review_biz",]

#Why this again?
if "category_diversity" in cell_features.columns:
    features.append("category_diversity")

X = cell_features[features].fillna(0)

#Look more into this
#z-scores Ex: n_businesses = 500 but pct_high_review_biz = 0.25
scaler = StandardScaler()
#turn raw nums into standardized z-scores and get mean of all scores of each cell
cell_features["urban_index"] = scaler.fit_transform(X).mean(axis=1)

In [ ]:
#split based on biz density
cell_features["urban_rural"] = pd.qcut(cell_features["urban_index"], q=2, labels=["rural", "urban"])

In [ ]:
#give each biz label
df_meta_cells = df_meta_cells.merge(cell_features[["geo_cell", "urban_rural", "urban_index"]], on="geo_cell", how="left")

#same thing
df_review = df_review.merge(df_meta_cells[["gmap_id", "urban_rural", "urban_index"]], on="gmap_id", how="left")


In [ ]:
df_meta_cells.groupby("urban_rural").agg(
    avg_rating=("avg_rating", "mean"),
    median_reviews=("num_of_reviews", "median"),
    pct_5star=("avg_rating", lambda x: (x >= 4.5).mean()), #review with Dhruv or Ansh
)


C:\Users\aiden\AppData\Local\Temp\ipykernel_27820\1569660553.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_meta_cells.groupby("urban_rural").agg(


,avg_rating,median_reviews,pct_5star
urban_rural,,,
rural,4.489671,8.0,0.653840
urban,4.262043,38.0,0.462294


In [ ]:
#Too many empty text on reviews
df_review["review_len"] = df_review["text"].fillna("").str.len()

df_review.groupby("urban_rural").agg(
    avg_review_rating=("rating", "mean"),
    avg_review_length=("review_len", "mean"),
    n_reviews=("gmap_id", "size"),
)

C:\Users\aiden\AppData\Local\Temp\ipykernel_27820\535703739.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_review.groupby("urban_rural").agg(


,avg_review_rating,avg_review_length,n_reviews
urban_rural,,,
rural,4.489554,98.592364,165086
urban,4.262017,99.413384,15816382
